# Fused Analog Kernel — Validation & Benchmarks

This notebook validates the Triton fused analog inference kernel and benchmarks it against the PyTorch baseline.

**Hardware target:** Colab T4 (free tier) or A100 (if available).

**What it covers:**
1. CPU smoke tests (imports, reference correctness).
2. GPU numerical correctness (Triton vs Python reference).
3. Speedup benchmarks (matmul + mismatch + noise + ADC + ReLU).
4. CUDA extension build (best-effort on Colab).
5. Substrate-aware mini-sweep (PCM / ReRAM / Capacitive).
6. Export results for local analysis.


---
## Cell 1 — Environment Setup

Install `triton`, `pytest`, and clone the `neuro-analog` repo.

In [ ]:
!pip install -q triton pytest
import os, sys, subprocess, json, math, time, textwrap, pathlib

REPO_URL = "https://github.com/apuroop-mutyala/neuro-analog.git"   # <-- change to your repo
REPO_DIR = pathlib.Path("neuro-analog")

if not REPO_DIR.exists():
    !git clone -q {REPO_URL}
%cd neuro-analog
else:
    %cd neuro-analog
    !git pull -q

# Add repo to Python path so imports work without `pip install -e`
sys.path.insert(0, str(pathlib.Path().cwd()))
print("Repo root:", pathlib.Path().cwd())


---
## Cell 2 — CPU Smoke Tests

Validate imports and the **Python reference** implementation on CPU tensors.
These tests do **not** need a GPU.

In [ ]:
import torch
from neuro_analog.kernels.triton.analog_ops import (
    analog_linear_reference, precompute_kernel_params
)

def test_import():
    assert analog_linear_reference is not None
    print("[PASS] import")

def test_reference_correctness():
    M, K = 64, 32
    x = torch.randn(K)
    w = torch.randn(M, K)
    bias = torch.randn(M)
    mismatch = torch.randn(M, K) * 0.01

    y_ref = analog_linear_reference(
        x, w, bias, mismatch, noise_sigma=0.05, adc_levels=256.0, activation="relu"
    )
    assert y_ref.shape == (M,)
    assert y_ref.dtype == torch.float32
    print("[PASS] reference_correctness")

def test_reference_none_mismatch():
    M, K = 64, 32
    x = torch.randn(K)
    w = torch.randn(M, K)
    bias = torch.randn(M)
    y = analog_linear_reference(x, w, bias, None, 0.0, 0.0, activation="none")
    assert y.shape == (M,)
    print("[PASS] reference_none_mismatch")

test_import()
test_reference_correctness()
test_reference_none_mismatch()
print("\n✅ CPU smoke tests passed.")


---
## Cell 3 — GPU Numerical Correctness (Triton)

Run the full `TestTritonOnGPU` test suite on a Colab GPU.

**Checks:**
- Deterministic no-noise match (no quant, no noise).
- Seeded noise + quantization match.
- Batched input (`B, K`).
- Large shape (`4096, 1024`) without OOM.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Change runtime to GPU (Runtime → Change runtime type → GPU).")

from neuro_analog.kernels.triton.analog_ops import analog_linear_fused

def test_fused_matches_reference_no_noise():
    M, K = 256, 128
    x = torch.randn(K, device="cuda")
    w = torch.randn(M, K, device="cuda")
    bias = torch.randn(M, device="cuda")
    mismatch = torch.randn(M, K, device="cuda") * 0.01

    y_ref = analog_linear_reference(
        x.cpu(), w.cpu(), bias.cpu(), mismatch.cpu(),
        noise_sigma=0.0, adc_levels=0.0, activation="relu"
    ).cuda()

    y_fused = analog_linear_fused(
        x, w, bias, mismatch, noise_sigma=0.0, adc_levels=0.0, activation="relu"
    )

    torch.testing.assert_close(y_ref, y_fused, atol=1e-3, rtol=1e-3)
    print("[PASS] fused_matches_reference_no_noise")

def test_fused_matches_reference_with_noise_and_quant():
    M, K = 256, 128
    x = torch.randn(K, device="cuda")
    w = torch.randn(M, K, device="cuda")
    bias = torch.randn(M, device="cuda")
    mismatch = torch.randn(M, K, device="cuda") * 0.01
    noise_sigma = 0.05
    adc_levels = 256.0

    torch.manual_seed(42)
    y_ref = analog_linear_reference(
        x.cpu(), w.cpu(), bias.cpu(), mismatch.cpu(),
        noise_sigma, adc_levels, activation="relu"
    ).cuda()

    torch.manual_seed(42)
    y_fused = analog_linear_fused(
        x, w, bias, mismatch, noise_sigma, adc_levels, activation="relu"
    )

    torch.testing.assert_close(y_ref, y_fused, atol=1e-3, rtol=1e-3)
    print("[PASS] fused_matches_reference_with_noise_and_quant")

def test_fused_batched():
    B, M, K = 8, 256, 128
    x = torch.randn(B, K, device="cuda")
    w = torch.randn(M, K, device="cuda")
    bias = torch.randn(M, device="cuda")
    mismatch = torch.randn(M, K, device="cuda") * 0.01

    y_ref = analog_linear_reference(
        x.cpu(), w.cpu(), bias.cpu(), mismatch.cpu(),
        noise_sigma=0.0, adc_levels=0.0, activation="relu"
    ).cuda()

    y_fused = analog_linear_fused(
        x, w, bias, mismatch, noise_sigma=0.0, adc_levels=0.0, activation="relu"
    )

    torch.testing.assert_close(y_ref, y_fused, atol=1e-3, rtol=1e-3)
    print("[PASS] fused_batched")

def test_fused_large_shape():
    M, K = 4096, 1024
    x = torch.randn(K, device="cuda")
    w = torch.randn(M, K, device="cuda")
    bias = torch.randn(M, device="cuda")
    y = analog_linear_fused(x, w, bias, None, 0.0, 0.0, activation="relu")
    assert y.shape == (M,)
    print("[PASS] fused_large_shape")

test_fused_matches_reference_no_noise()
test_fused_matches_reference_with_noise_and_quant()
test_fused_batched()
test_fused_large_shape()
print("\n✅ GPU correctness tests passed.")


---
## Cell 4 — Benchmarks vs PyTorch Baseline

Run `benchmarks/bench_triton.py` shapes: `(1024,512)`, `(4096,1024)`, `(16384,4096)` with batch=1 and batch=8.

Results are saved to `benchmarks/results/triton_bench.json`.

In [ ]:
from benchmarks.bench_triton import benchmark_shape

shapes = [
    (1024, 512, 1),
    (1024, 512, 8),
    (4096, 1024, 1),
    (4096, 1024, 8),
    (16384, 4096, 1),
    (16384, 4096, 8),
]

results = []
for M, K, B in shapes:
    print(f"Benchmarking M={M}, K={K}, B={B} ...")
    try:
        r = benchmark_shape(M, K, B, repeats=100, warmup=10)
        results.append(r)
        print(
            f"  Baseline: {r['baseline_ms']:.3f} ms  |  "
            f"Fused: {r['fused_ms']:.3f} ms  |  "
            f"Speedup: {r['speedup']:.2f}x"
        )
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({"M": M, "K": K, "B": B, "error": str(e)})

# Save
out_dir = pathlib.Path("benchmarks/results")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "triton_bench.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n📁 Results saved to {out_path}")


---
## Cell 5 — CUDA Extension Build (Best-Effort)

Colab T4 ships with CUDA 12.x but the `nvcc` version can be stale.
This cell tries to build the C++ extension; if it fails we document the error and skip.

In [ ]:
cuda_build_log = pathlib.Path("benchmarks/results/cuda_build.log")
cuda_ok = False

try:
    !cd neuro_analog/kernels/cuda && pip install -e . > {cuda_build_log} 2>&1
    import neuro_analog_cuda
    cuda_ok = True
    print("✅ CUDA extension built and imported successfully.")
except Exception as e:
    print(f"⚠️ CUDA build failed (common on Colab): {e}")
    print(f"   Log written to {cuda_build_log}")

if cuda_ok:
    # Quick CUDA-vs-Triton numerical comparison
    M, K = 256, 128
    x = torch.randn(K, device="cuda")
    w = torch.randn(M, K, device="cuda")
    bias = torch.randn(M, device="cuda")
    mismatch = torch.randn(M, K, device="cuda") * 0.01

    y_triton = analog_linear_fused(x, w, bias, mismatch, 0.0, 0.0, activation="relu")
    y_cuda = neuro_analog_cuda.analog_linear_fused_cuda(x, w, bias, mismatch, 0.0, 0.0)

    torch.testing.assert_close(y_triton, y_cuda, atol=1e-3, rtol=1e-3)
    print("[PASS] CUDA vs Triton numerical match.")


---
## Cell 6 — Substrate-Aware Mini-Sweep

Validate that the fused kernel matches the Python reference across **PCM**, **ReRAM**, and **Capacitive** substrates.

We run a tiny sweep (5 sigma values × 3 trials) and verify the outputs match within `atol=1e-3`.
We also report the wall-clock reduction vs the Python sequential implementation.

In [ ]:
from neuro_analog.simulator.substrates import (
    get_substrate, PCMSubstrate, ReRAMSubstrate, CapacitiveSubstrate
)
from neuro_analog.simulator.analog_linear import AnalogLinear
from neuro_analog.kernels.integration import AnalogLinearFused
import time

def mini_sweep(substrate_name: str, M=256, K=128, trials=3, sigmas=[0.0, 0.02, 0.05, 0.10, 0.15]):
    """Run a tiny mismatch sweep and compare fused vs reference."""
    substrate = get_substrate(substrate_name) if substrate_name else None
    w = torch.randn(M, K, device="cuda")
    b = torch.randn(M, device="cuda")

    results = []
    for sigma in sigmas:
        for t in range(trials):
            layer = AnalogLinear(
                in_features=K, out_features=M,
                weight=w, bias=b,
                sigma_mismatch=sigma,
                substrate=substrate
            ).cuda()

            fused = AnalogLinearFused.from_analog_linear(layer, substrate)
            fused = fused.cuda()

            x = torch.randn(K, device="cuda")

            # Python reference (no backend dispatch — just call layer directly)
            torch.manual_seed(42 + t)
            y_ref = layer(x)

            # Fused kernel (with seed alignment for noise)
            torch.manual_seed(42 + t)
            y_fused = fused(x)

            torch.testing.assert_close(y_ref, y_fused, atol=1e-3, rtol=1e-3)
            results.append({"sigma": sigma, "trial": t, "match": True})

    print(f"  {substrate_name or 'none':12s}: {len(results)} / {len(results)} matched.")
    return results

print("Substrate mini-sweep (fused vs reference):")
for sub in ["pcm", "reram", "capacitive", None]:
    mini_sweep(sub, M=256, K=128, trials=3, sigmas=[0.0, 0.02, 0.05])
print("\n✅ Substrate sweep validation passed.")


---
## Cell 7 — Export & Download Results

Tar the `benchmarks/results/` directory and provide a download link.

In [ ]:
import base64, tarfile, io

results_dir = pathlib.Path("benchmarks/results")
tar_path = pathlib.Path("benchmarks/results/fused_kernel_results.tar.gz")

with tarfile.open(tar_path, "w:gz") as tar:
    for f in results_dir.glob("*.json"):
        tar.add(f, arcname=f.name)
    for f in results_dir.glob("*.log"):
        tar.add(f, arcname=f.name)

print(f"📦 Tar archive: {tar_path} ({tar_path.stat().st_size / 1024:.1f} KB)")

# Base64 for easy copy-paste out of Colab
with open(tar_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

print("\n--- Base64 encoded tar (first 200 chars) ---")
print(b64[:200] + "...")

# Google Colab download helper
from google.colab import files
files.download(str(tar_path))
